# Cluster 3 Usage: Testing the Exported Models

This notebook demonstrates how to load and use the preprocessing pipeline and stacking model that were exported from the Cluster 3 analysis.

## Setup

In [77]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import confusion_matrix, classification_report, recall_score, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully")

Libraries imported successfully


In [78]:
from sklearn.base import BaseEstimator, TransformerMixin

# Define ColumnDropper transformer (required to unpickle the saved pipeline)
class ColumnDropper(BaseEstimator, TransformerMixin):
    """Drops specified columns. Used as the first step of the pipeline so test data
    goes through the same column-drop step as training data."""
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=[c for c in self.columns_to_drop if c in X.columns])
        return X

## Load the Preprocessing Pipeline and Stacking Model

In [79]:
# Load the saved pipeline and model
preprocessing_pipeline = joblib.load('cluster_3_preprocessing_pipeline.joblib')
stacking_model = joblib.load('cluster_3_stacking_model.joblib')

print("✓ Preprocessing pipeline loaded")
print(f"  Pipeline steps: {[name for name, _ in preprocessing_pipeline.steps]}")
print()
print("✓ Stacking model loaded")
print(f"  Model type: {type(stacking_model).__name__}")
print(f"  Base learners: {[name for name, _ in stacking_model.estimators]}")

✓ Preprocessing pipeline loaded
  Pipeline steps: ['drop_cols', 'scaler', 'pca']

✓ Stacking model loaded
  Model type: StackingClassifier
  Base learners: ['gb', 'svc', 'rf', 'lr']


In [80]:
# Inspect pipeline internals to understand what it expects
print("\n=== Pipeline Internals ===")
for name, transformer in preprocessing_pipeline.named_steps.items():
    print(f"\n{name}:")
    if hasattr(transformer, 'n_features_in_'):
        print(f"  n_features_in_: {transformer.n_features_in_}")
    if hasattr(transformer, 'n_features_out_'):
        print(f"  n_features_out_: {transformer.n_features_out_}")
    if hasattr(transformer, 'columns_to_drop'):
        print(f"  columns_to_drop: {transformer.columns_to_drop}")
    if hasattr(transformer, 'feature_names_in_'):
        print(f"  feature_names_in_: {list(transformer.feature_names_in_) if len(transformer.feature_names_in_) < 20 else f'{len(transformer.feature_names_in_)} features'}")


=== Pipeline Internals ===

drop_cols:
  columns_to_drop: ['Index', 'Liability-Assets Flag', 'Net Income Flag', 'Cluster']

scaler:
  n_features_in_: 14
  feature_names_in_: ['Operating Expense Rate', 'Research and development expense rate', 'Interest-bearing debt interest rate', 'Total Asset Growth Rate', 'Quick Ratio', 'Inventory Turnover Rate (times)', 'Allocation rate per person', 'Cash/Current Liability', 'Inventory/Current Liability', 'Long-term Liability to Current Assets', 'Current Asset Turnover Rate', 'Quick Asset Turnover Rate', 'Cash Turnover Rate', 'Total assets to GNP price']

pca:
  n_features_in_: 93
  feature_names_in_: 93 features


## Load Test Data

In [81]:
# Load the cluster 3 data
df = pd.read_csv('../../Clusters/cluster_3.csv')

## Preprocess the Data

In [82]:
# Separate features and target
X = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

print(f"Original features shape: {X.shape}")

# Apply preprocessing pipeline step by step to preserve column order
# (The pipeline has a structural issue: scaler expects 14 columns but dropper outputs 93)

# Step 1: Drop columns
columns_to_drop = ['Index', 'Liability-Assets Flag', 'Net Income Flag', 'Cluster']
X_dropped = X.drop(columns=[c for c in columns_to_drop if c in X.columns])
print(f"After dropping columns: {X_dropped.shape}")

# Step 2: Get the exact columns the scaler was fitted on
scaler_columns = list(preprocessing_pipeline.named_steps['scaler'].feature_names_in_)
print(f"Scaler will transform {len(scaler_columns)} columns")

# Step 3: Apply the scaler to those specific columns (preserves original column order)
X_scaled = X_dropped.copy()
X_scaled[scaler_columns] = preprocessing_pipeline.named_steps['scaler'].transform(X_dropped[scaler_columns])
print(f"After scaling: {X_scaled.shape}")

# Step 4: Apply PCA (pass DataFrame to preserve feature names)
X_preprocessed = preprocessing_pipeline.named_steps['pca'].transform(X_scaled)

print(f"Preprocessed features shape: {X_preprocessed.shape}")
print(f"\nPreprocessing steps applied:")
print(f"  1. Dropped: {columns_to_drop}")
print(f"  2. Scaled {len(scaler_columns)} columns using MinMaxScaler")
print(f"  3. Reduced to {X_preprocessed.shape[1]} components using PCA")

Original features shape: (1024, 97)
After dropping columns: (1024, 93)
Scaler will transform 14 columns
After scaling: (1024, 93)
Preprocessed features shape: (1024, 8)

Preprocessing steps applied:
  1. Dropped: ['Index', 'Liability-Assets Flag', 'Net Income Flag', 'Cluster']
  2. Scaled 14 columns using MinMaxScaler
  3. Reduced to 8 components using PCA


## Make Predictions

In [83]:
# Make predictions
y_pred = stacking_model.predict(X_preprocessed)
y_pred_proba = stacking_model.predict_proba(X_preprocessed)

print(f"Predictions made for {len(y_pred)} samples")
print(f"\nPrediction distribution:")
print(f"  Non-Bankrupt (0): {(y_pred == 0).sum()}")
print(f"  Bankrupt (1): {(y_pred == 1).sum()}")

Predictions made for 1024 samples

Prediction distribution:
  Non-Bankrupt (0): 630
  Bankrupt (1): 394
